# Matched combinatorial-topology benchmark

This analysis compares two libraries containing the **same matched compound IDs** but different attachment geometries (L1 and L2). The question is whether COAF distinguishes the topology change more clearly than ECFP.

Three complementary readouts are used:

1. **Matched L1–L2 distance:** for each ID, compare its two topology variants.
2. **Controlled topology separation:** for the same sampled pair of IDs, compare within-topology distances with cross-topology distances.
3. **Nearest-neighbor enrichment:** test whether nearest neighbors preferentially come from the same topology.

> Important correction: the exploratory notebook used a directed path fingerprint in the cells labeled COAF. This notebook explicitly calls the finalized `coaf_fingerprint_from_smiles` implementation in `coaf.py`.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'coaf.py').exists():
    raise FileNotFoundError('Start Jupyter in the COAF project folder containing coaf.py.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import validation.matched_topology as matched_topology
matched_topology = importlib.reload(matched_topology)
MatchedTopologyConfig = matched_topology.MatchedTopologyConfig
load_matched_topology_dataset = matched_topology.load_matched_topology_dataset
run_matched_topology_benchmark = matched_topology.run_matched_topology_benchmark

plt.style.use('default')
pd.set_option('display.max_columns', 30)

## 1. Input and output locations

The expected input columns are `ID`, `ECFP_L1`, `ECFP_L2`, `COAF_L1`, and `COAF_L2`. The ECFP columns contain ordinary structures; the COAF columns contain exactly one `[Hg]` root marker. Change `DATA_FILE` if the CSV has a different filename.

The results folder is deliberately run-specific. The analysis will not overwrite a nonempty folder.

In [ ]:
DATA_FILE = PROJECT_DIR / 'data' / 'LibraryComparison.csv'
RESULT_DIR = PROJECT_DIR / 'results' / 'matched_topology_all_descriptors_r3_bits1024_temp'
FIGURE_DIR = RESULT_DIR / 'figures'

print('Input:', DATA_FILE)
print('Output:', RESULT_DIR)

## 2. Validate the matched dataset

This checks the required columns, unique IDs, valid SMILES, and one `[Hg]` marker in every rooted structure. The preview confirms which records and representations will be compared.

In [ ]:
dataset = load_matched_topology_dataset(DATA_FILE)
print(f'{len(dataset):,} matched IDs loaded.')
display(dataset.head())

## 3. Run the benchmark

Both fingerprints use 1,024 bits and radius 3. The controlled analysis samples unique unordered ID pairs once and reuses those exact pairs for ECFP and COAF. The neighborhood analysis uses a reproducible subset to keep its all-versus-all calculation manageable.

In [ ]:
config = MatchedTopologyConfig(
    n_bits=1024,
    ecfp_radius=3,
    coaf_radius=3,
    poaf_max_path_length=6,
    n_pair_samples=100_000,
    neighbor_subset_ids=2_000,
    neighbor_k=(1, 5, 10, 20),
    ranking_size=20,
    random_seed=123,
    population_component_counts=(1, 2, 3, 4),
    population_primary_components=2,
)

results = run_matched_topology_benchmark(
    dataset,
    config=config,
    output_dir=RESULT_DIR,
)

### Dataset and fingerprint audit

`dataset_summary` records the analysis scale. `fingerprint_summary` reports the number of set bits and bit density for each descriptor/topology combination; large density differences should be considered when interpreting Tanimoto distances.

In [ ]:
display(results.dataset_summary)
display(results.fingerprint_summary.round(4))

## 4. Matched L1–L2 topology distance

The contour plot summarizes the joint distribution of matched compound IDs as a function of their ECFP and COAF L1–L2 distances. Density is estimated by a two-dimensional Gaussian kernel density estimate using Scott’s bandwidth rule. The contours enclose 50%, 75%, 90%, and 95% of the estimated probability mass. Regions above the diagonal contain compounds for which COAF assigns a larger topology distance; regions below it favor ECFP. The difference distribution summarizes the direction and magnitude across the complete matched library and uses the data-dependent Freedman–Diaconis histogram rule.

In [ ]:
display(results.matched_summary.round(4))

def save_figure(fig, filename, png_dpi=600):
    """Save a quantitative figure as editable SVG and high-resolution PNG."""
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    stem = Path(filename).stem
    svg_path = FIGURE_DIR / f'{stem}.svg'
    png_path = FIGURE_DIR / f'{stem}.png'
    fig.savefig(svg_path, format='svg', bbox_inches='tight')
    fig.savefig(png_path, format='png', dpi=png_dpi, bbox_inches='tight')
    print(f'Saved figure: {svg_path}')
    print(f'Saved figure: {png_path}')
    return {'svg': svg_path, 'png': png_path}

def plot_matched_topology_density(table):
    from scipy.stats import gaussian_kde

    x = table['ECFP_distance_L1_L2'].to_numpy(dtype=float)
    y = table['COAF_distance_L1_L2'].to_numpy(dtype=float)
    limits = (0, max(float(x.max()), float(y.max())) * 1.03)
    fig, ax = plt.subplots(figsize=(6.2, 5.0))

    # Estimate the two-dimensional density using Scott's bandwidth rule.
    kde = gaussian_kde(np.vstack([x, y]), bw_method='scott')
    grid = np.linspace(limits[0], limits[1], 160)
    grid_x, grid_y = np.meshgrid(grid, grid)
    density = kde(np.vstack([grid_x.ravel(), grid_y.ravel()])).reshape(grid_x.shape)

    # Determine density thresholds enclosing fixed fractions of total KDE mass.
    probability_masses = (0.50, 0.75, 0.90, 0.95)
    ordered_density = np.sort(density.ravel())[::-1]
    cumulative_mass = np.cumsum(ordered_density)
    cumulative_mass /= cumulative_mass[-1]
    thresholds = {
        mass: ordered_density[min(np.searchsorted(cumulative_mass, mass), len(ordered_density) - 1)]
        for mass in probability_masses
    }

    colors = plt.cm.viridis(np.linspace(0.15, 0.90, len(probability_masses)))
    for mass, color in zip(probability_masses, colors):
        contour = ax.contour(
            grid_x, grid_y, density, levels=[thresholds[mass]],
            colors=[color], linewidths=1.8,
        )
        ax.clabel(
            contour, fmt={thresholds[mass]: f'{int(mass * 100)}%'},
            inline=True, fontsize=8,
        )
    ax.plot(limits, limits, '--', color='black', linewidth=1)
    ax.set(xlabel='ECFP L1–L2 distance', ylabel='COAF L1–L2 distance',
                xlim=limits, ylim=limits,
                title='Matched topology-distance probability regions')
    ax.text(
        0.03, 0.97, "Contours enclose 50%, 75%, 90%, and 95%\nof the KDE probability mass",
        transform=ax.transAxes, va='top', fontsize=8,
        bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none'},
    )
    fig.tight_layout()
    return fig

fig = plot_matched_topology_density(results.matched_pair_distances)
save_figure(fig, "matched_topology_distance_density.svg")
fig.savefig(
    "matched_topology_distance_density.png",
    dpi=300,
    bbox_inches="tight",
)

### Descriptor difference for each matched ID

For each chemical ID, this analysis compares its matched L1 and L2 representations. The plotted value is $d_{\mathrm{COAF}}(L1_i,L2_i)-d_{\mathrm{ECFP}}(L1_i,L2_i)$. Positive values mean that COAF distinguishes the two attachment geometries more strongly than ECFP for that matched compound.

In [ ]:
def plot_matched_id_descriptor_difference(table):
    values = table['COAF_minus_ECFP_distance'].dropna()
    fig, ax = plt.subplots(figsize=(6.2, 4.5))
    ax.hist(values, bins='fd', color='#3b82f6', alpha=0.85)
    ax.axvline(0, linestyle='--', color='black', linewidth=1, label='No descriptor difference')
    ax.axvline(values.median(), linestyle=':', color='#1d4ed8', linewidth=1.5, label='Median')
    ax.set(xlabel='COAF distance − ECFP distance', ylabel='Matched compound count',
           title='Descriptor difference for each matched ID')
    ax.legend(frameon=False)
    fig.tight_layout()
    return fig

fig = plot_matched_id_descriptor_difference(results.matched_pair_distances)
save_figure(fig, 'matched_ID_descriptor_difference.svg');

In [ ]:
print('Mixture-model comparison (lower AIC/BIC is better)')
display(results.population_model_comparison.round(2))
print('Two-component descriptive population estimate')
display(results.population_component_summary.round(4))

## 5. Controlled within-versus-cross topology separation

For two different IDs *i* and *j*, the within-topology distance is the mean of L1ᵢ–L1ⱼ and L2ᵢ–L2ⱼ. The cross-topology distance is the mean of L1ᵢ–L2ⱼ and L2ᵢ–L1ⱼ. Their difference is the **topology separation**. Positive values mean topology changes increase distance beyond the ordinary chemical difference between the two IDs.

The same ID pairs are used for both descriptors, making the COAF–ECFP comparison paired and controlled. In the right-hand plot, each hexagonal bin contains sampled ID pairs with similar ECFP and COAF topology-separation values. The color reports how many sampled pairs fall in that bin. A value of zero means that changing topology does not add separation beyond the ordinary difference between IDs *i* and *j*. Positive values indicate topology-sensitive separation. Bins above the diagonal indicate greater topology separation with COAF; bins below the diagonal indicate greater separation with ECFP.

In [ ]:
display(results.topology_separation_summary.round(4))
display(results.descriptor_comparison.round(4))
def prepare_pair_type_distances(table):
    """
    Convert the controlled-pair results into a tidy table containing
    the ECFP and COAF distances for the three pair types.
    """
    column_labels = {
        'within_L1_distance': 'Within L1',
        'within_L2_distance': 'Within L2',
        'cross_mean_distance': 'Across L1/L2',
    }

    parts = []

    for column, pair_type in column_labels.items():
        subset = table[
            ['pair_number', 'descriptor', column]
        ].copy()

        subset = subset.rename(columns={column: 'distance'})
        subset['pair_type'] = pair_type
        parts.append(subset)

    return pd.concat(parts, ignore_index=True)


def prepare_pair_type_descriptor_differences(pair_type_distances):
    """
    Calculate COAF distance minus ECFP distance for each sampled ID pair
    and each pair type.
    """
    wide = pair_type_distances.pivot(
        index=['pair_number', 'pair_type'],
        columns='descriptor',
        values='distance',
    )

    differences = (
        wide['COAF'] - wide['ECFP']
    ).rename('COAF_minus_ECFP_distance').reset_index()

    return differences


def summarize_pair_type_distances(pair_type_distances):
    """
    Summarize the distance distributions separately for ECFP and COAF.
    """
    return (
        pair_type_distances
        .groupby(['descriptor', 'pair_type'], sort=False)['distance']
        .agg(n='count', mean='mean', median='median', std='std')
        .reset_index()
    )


def summarize_pair_type_descriptor_differences(differences):
    """
    Summarize COAF-minus-ECFP distance differences by pair type.
    """
    return (
        differences
        .groupby('pair_type', sort=False)['COAF_minus_ECFP_distance']
        .agg(n='count', mean='mean', median='median', std='std')
        .reset_index()
    )


def plot_pair_type_distances(pair_type_distances, descriptor):
    """
    Plot the distance distributions for one descriptor.

    Each curve integrates to approximately 1 and therefore represents
    probability density rather than compound or pair counts.
    """
    from scipy.stats import gaussian_kde

    order = ['Within L1', 'Within L2', 'Across L1/L2']

    colors = {
        'Within L1': '#f97316',
        'Within L2': '#10b981',
        'Across L1/L2': '#3b82f6',
    }

    descriptor_data = pair_type_distances.loc[
        pair_type_distances['descriptor'] == descriptor
    ]

    # Tanimoto distances are bounded between 0 and 1.
    grid = np.linspace(0, 1, 500)

    fig, ax = plt.subplots(figsize=(7.2, 4.8))

    for pair_type in order:
        values = descriptor_data.loc[
            descriptor_data['pair_type'] == pair_type,
            'distance',
        ].dropna().to_numpy()

        density = gaussian_kde(
            values,
            bw_method='scott',
        )(grid)

        ax.plot(
            grid,
            density,
            color=colors[pair_type],
            linewidth=2,
            label=pair_type,
        )

    ax.set(
        xlim=(0, 1),
        xlabel=f'{descriptor} Tanimoto distance',
        ylabel='Probability density',
        title=f'{descriptor} distance distributions by pair type',
    )

    ax.legend(frameon=False)
    fig.tight_layout()

    return fig


def plot_pair_type_descriptor_differences(differences):
    """
    Plot COAF-minus-ECFP distance distributions for the three pair types.
    """
    from scipy.stats import gaussian_kde

    order = ['Within L1', 'Within L2', 'Across L1/L2']

    colors = {
        'Within L1': '#f97316',
        'Within L2': '#10b981',
        'Across L1/L2': '#3b82f6',
    }

    values = differences[
        'COAF_minus_ECFP_distance'
    ].dropna().to_numpy()

    margin = max(
        0.02,
        0.03 * (values.max() - values.min()),
    )

    grid = np.linspace(
        values.min() - margin,
        values.max() + margin,
        500,
    )

    fig, ax = plt.subplots(figsize=(7.2, 4.8))

    for pair_type in order:
        group = differences.loc[
            differences['pair_type'] == pair_type,
            'COAF_minus_ECFP_distance',
        ].dropna().to_numpy()

        density = gaussian_kde(
            group,
            bw_method='scott',
        )(grid)

        ax.plot(
            grid,
            density,
            color=colors[pair_type],
            linewidth=2,
            label=pair_type,
        )

    ax.axvline(
        0,
        linestyle='--',
        color='black',
        linewidth=1,
        label='No descriptor difference',
    )

    ax.set(
        xlabel='COAF distance − ECFP distance',
        ylabel='Probability density',
        title='Descriptor-distance differences by pair type',
    )

    ax.legend(frameon=False)
    fig.tight_layout()

    return fig


# Prepare the distance and difference tables.
pair_type_distances = prepare_pair_type_distances(
    results.controlled_pair_distances
)

pair_type_differences = prepare_pair_type_descriptor_differences(
    pair_type_distances
)


# Display descriptive statistics for the individual descriptors.
print('Tanimoto distances for ECFP and COAF')
display(
    summarize_pair_type_distances(
        pair_type_distances
    ).round(4)
)


# Display descriptive statistics for COAF minus ECFP.
print('COAF distance minus ECFP distance')
display(
    summarize_pair_type_descriptor_differences(
        pair_type_differences
    ).round(4)
)


# Plot and save the ECFP distance distributions.
fig = plot_pair_type_distances(
    pair_type_distances,
    descriptor='ECFP',
)

save_figure(
    fig,
    'ECFP_within_and_cross_distance_distributions.svg',
)


# Plot and save the COAF distance distributions.
fig = plot_pair_type_distances(
    pair_type_distances,
    descriptor='COAF',
)

save_figure(
    fig,
    'COAF_within_and_cross_distance_distributions.svg',
)


# Display the descriptive statistics for ECFP6-RN and POAF6.
additional_descriptor_summary = (
    summarize_pair_type_distances(pair_type_distances)
    .loc[
        lambda table: table['descriptor'].isin(
            ['ECFP_Hg', 'POAF']
        )
    ]
    .copy()
)

additional_descriptor_summary['descriptor'] = (
    additional_descriptor_summary['descriptor']
    .replace({
        'ECFP_Hg': 'ECFP6-RN',
        'POAF': 'POAF6',
    })
)

display(additional_descriptor_summary.round(4))


# ECFP6-RN within- and cross-library distributions.
fig = plot_pair_type_distances(
    pair_type_distances,
    descriptor='ECFP_Hg',
)

fig.axes[0].set(
    xlabel='ECFP6-RN Tanimoto distance',
    title='ECFP6-RN distance distributions by pair type',
)

save_figure(
    fig,
    'ECFP6_RN_within_and_cross_distance_distributions',
)


# POAF6 within- and cross-library distributions.
fig = plot_pair_type_distances(
    pair_type_distances,
    descriptor='POAF',
)

fig.axes[0].set(
    xlabel='POAF6 Tanimoto distance',
    title='POAF6 distance distributions by pair type',
)

save_figure(
    fig,
    'POAF6_within_and_cross_distance_distributions',
)

# Plot and save the paired COAF-minus-ECFP differences.
fig = plot_pair_type_descriptor_differences(
    pair_type_differences
)

save_figure(
    fig,
    'within_and_cross_descriptor_difference_distributions.svg',
)
def plot_controlled_separation(table):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    groups = [table.loc[table['descriptor'] == name, 'topology_separation'].to_numpy()
              for name in ('ECFP', 'COAF')]
    axes[0].violinplot(groups, positions=[1, 2], showmedians=True, showextrema=True)
    axes[0].set_xticks([1, 2], ['ECFP', 'COAF'])
    axes[0].axhline(0, linestyle='--', color='black', linewidth=1)
    axes[0].set(title='Controlled topology-separation distribution',
                ylabel='Cross distance − within distance', xlabel='')
    wide = table.pivot(index='pair_number', columns='descriptor', values='topology_separation')
    hexbin = axes[1].hexbin(
        wide['ECFP'], wide['COAF'], gridsize=45, mincnt=1,
        cmap='viridis', bins='log',
    )
    fig.colorbar(hexbin, ax=axes[1], label='Sampled ID-pair count (log scale)')
    low = min(float(wide.min().min()), 0); high = max(float(wide.max().max()), 0)
    axes[1].plot([low, high], [low, high], '--', color='black', linewidth=1)
    axes[1].set(xlabel='ECFP topology separation', ylabel='COAF topology separation',
                title='Same sampled ID pairs')
    axes[1].text(
        0.03, 0.97, 'Above diagonal: greater COAF separation',
        transform=axes[1].transAxes, va='top', fontsize=9,
        bbox={'facecolor': 'white', 'alpha': 0.75, 'edgecolor': 'none'},
    )
    plt.tight_layout()
    return fig

fig = plot_controlled_separation(results.controlled_pair_distances)
save_figure(fig, 'controlled_topology_separation.svg');

### COAF–ECFP distance differences within and across libraries

The same sampled pairs of distinct chemical IDs are evaluated three ways: both structures from L1, both from L2, and across L1/L2. For the cross-library comparison, the two possible directions (L1ᵢ–L2ⱼ and L2ᵢ–L1ⱼ) are averaged. Each curve shows $d_{\mathrm{COAF}}-d_{\mathrm{ECFP}}$; positive values indicate that COAF assigns the pair a larger Tanimoto distance. A selective positive shift for the cross-library curve would support attachment-topology discrimination rather than a general tendency for COAF to produce larger distances.

In [ ]:
def prepare_pair_type_descriptor_differences(table):
    # Put ECFP and COAF results for each sampled ID pair on one row.
    distance_columns = [
        'within_L1_distance', 'within_L2_distance', 'cross_mean_distance'
    ]
    wide = table.pivot(index='pair_number', columns='descriptor', values=distance_columns)

    labels = {
        'within_L1_distance': 'Within L1',
        'within_L2_distance': 'Within L2',
        'cross_mean_distance': 'Across L1/L2',
    }
    parts = []
    for column, label in labels.items():
        difference = wide[(column, 'COAF')] - wide[(column, 'ECFP')]
        parts.append(pd.DataFrame({
            'pair_number': difference.index,
            'pair_type': label,
            'COAF_minus_ECFP_distance': difference.to_numpy(),
        }))
    return pd.concat(parts, ignore_index=True)

def summarize_pair_type_descriptor_differences(differences):
    return (differences.groupby('pair_type', sort=False)['COAF_minus_ECFP_distance']
            .agg(n='count', mean='mean', median='median', std='std')
            .reset_index())

def plot_pair_type_descriptor_differences(differences):
    from scipy.stats import gaussian_kde

    order = ['Within L1', 'Within L2', 'Across L1/L2']
    colors = {'Within L1': '#f97316', 'Within L2': '#10b981',
              'Across L1/L2': '#3b82f6'}
    values = differences['COAF_minus_ECFP_distance'].dropna().to_numpy()
    margin = max(0.02, 0.03 * (values.max() - values.min()))
    grid = np.linspace(values.min() - margin, values.max() + margin, 500)

    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    for label in order:
        group = differences.loc[
            differences['pair_type'] == label, 'COAF_minus_ECFP_distance'
        ].dropna().to_numpy()
        density = gaussian_kde(group, bw_method='scott')(grid)
        ax.plot(grid, density, color=colors[label], linewidth=2, label=label)

    ax.axvline(0, linestyle='--', color='black', linewidth=1,
               label='No descriptor difference')
    ax.set(xlabel='COAF distance − ECFP distance', ylabel='Probability density',
           title='Descriptor-distance differences by pair type')
    ax.legend(frameon=False)
    fig.tight_layout()
    return fig

pair_type_differences = prepare_pair_type_descriptor_differences(
    results.controlled_pair_distances
)
display(summarize_pair_type_descriptor_differences(pair_type_differences).round(4))
fig = plot_pair_type_descriptor_differences(pair_type_differences)
save_figure(fig, 'within_and_cross_descriptor_difference_distributions.svg');

In [ ]:
from scipy.stats import gaussian_kde


def summarize_matched_descriptor_differences(table):
    """Summarize matched L1–L2 distance differences relative to ECFP6."""
    specifications = [
        (
            'ECFP6-RN',
            'ECFP_Hg_minus_ECFP_distance',
        ),
        (
            'COAF6',
            'COAF_minus_ECFP_distance',
        ),
        (
            'POAF6',
            'POAF_minus_ECFP_distance',
        ),
    ]

    records = []

    for descriptor, column in specifications:
        values = table[column].dropna().to_numpy(dtype=float)

        records.append({
            'descriptor': descriptor,
            'n_matched_compounds': len(values),
            'mean_difference': values.mean(),
            'median_difference': np.median(values),
            'std_difference': values.std(ddof=1),
            'fraction_greater_than_ECFP6': np.mean(values > 0),
        })

    return pd.DataFrame(records)


def plot_matched_descriptor_difference_density(
    table,
    bandwidth='scott',
):
    """
    Plot KDE probability densities for matched L1–L2 distance differences
    relative to standard ECFP6.
    """
    specifications = [
        (
            'ECFP6-RN',
            'ECFP_Hg_minus_ECFP_distance',
            '#059669',
        ),
        (
            'COAF6',
            'COAF_minus_ECFP_distance',
            '#2563eb',
        ),
        (
            'POAF6',
            'POAF_minus_ECFP_distance',
            '#7c3aed',
        ),
    ]

    required_columns = [
        column for _, column, _ in specifications
    ]

    missing = [
        column
        for column in required_columns
        if column not in table.columns
    ]

    if missing:
        raise ValueError(
            'The matched-pair table is missing required columns: '
            f'{missing}'
        )

    descriptor_values = {}

    for descriptor, column, _ in specifications:
        values = (
            table[column]
            .dropna()
            .to_numpy(dtype=float)
        )

        if len(values) < 2:
            raise ValueError(
                f'At least two values are required for {descriptor}.'
            )

        descriptor_values[descriptor] = values

    pooled = np.concatenate(
        list(descriptor_values.values())
    )

    data_range = pooled.max() - pooled.min()
    margin = max(0.02, 0.05 * data_range)

    grid = np.linspace(
        pooled.min() - margin,
        pooled.max() + margin,
        600,
    )

    fig, ax = plt.subplots(figsize=(7.4, 4.8))

    for descriptor, _, color in specifications:
        values = descriptor_values[descriptor]

        density = gaussian_kde(
            values,
            bw_method=bandwidth,
        )(grid)

        ax.plot(
            grid,
            density,
            color=color,
            linewidth=2.2,
            label=descriptor,
        )

        ax.fill_between(
            grid,
            0,
            density,
            color=color,
            alpha=0.08,
        )

    ax.axvline(
        0,
        color='black',
        linestyle='--',
        linewidth=1.2,
        label='Equal to ECFP6',
    )

    ax.set(
        xlabel=(
            'Descriptor L1–L2 distance − '
            'ECFP6 L1–L2 distance'
        ),
        ylabel='Probability density',
        title=(
            'Matched topology-distance differences '
            'relative to ECFP6'
        ),
    )

    ax.legend(
        frameon=False,
        ncol=2,
    )

    fig.tight_layout()

    return fig


display(
    summarize_matched_descriptor_differences(
        results.matched_pair_distances
    ).round(4)
)

fig = plot_matched_descriptor_difference_density(
    results.matched_pair_distances
)

save_figure(
    fig,
    'matched_topology_descriptor_differences_relative_to_ECFP6',
)

## 6. Same-topology nearest-neighbor enrichment

Each L1 or L2 structure is used as a query. The plotted value is the fraction of its top-*k* neighbors belonging to the same topology. A random topology-balanced ranking has an expected fraction near 0.5; values above 0.5 indicate that the descriptor organizes the library by attachment geometry.

In [ ]:
display(results.neighbor_summary.round(4))


def plot_neighbor_enrichment(summary):
    """
    Plot same-topology enrichment for four fingerprints at k=5 and k=10.

    Bars show mean same-topology neighbor fractions. Error bars show the
    standard deviation across query structures, not confidence intervals.
    """
    selected_k = [5, 10]
    descriptors = ["ECFP", "ECFP_Hg", "COAF", "POAF"]

    colors = {
        "ECFP": "#f97316",
        "ECFP_Hg": "#a855f7",
        "COAF": "#3b82f6",
        "POAF": "#10b981",
    }

    subset = summary.loc[
        summary["k"].isin(selected_k)
        & summary["descriptor"].isin(descriptors)
    ].copy()

    expected_rows = len(selected_k) * len(descriptors)
    if len(subset) != expected_rows:
        available = sorted(summary["descriptor"].dropna().unique())
        raise ValueError(
            f"The summary must contain one row for each of {descriptors} "
            f"at k=5 and k=10. Available descriptors: {available}"
        )

    x = np.arange(len(selected_k))
    width = 0.19
    n_descriptors = len(descriptors)

    fig, ax = plt.subplots(figsize=(8.5, 4.8))

    for index, descriptor in enumerate(descriptors):
        descriptor_data = (
            subset.loc[subset["descriptor"] == descriptor]
            .set_index("k")
            .loc[selected_k]
        )

        values = descriptor_data[
            "mean_same_topology_fraction"
        ].to_numpy()

        errors = descriptor_data[
            "std_same_topology_fraction"
        ].to_numpy()

        offset = (index - (n_descriptors - 1) / 2) * width
        positions = x + offset

        bars = ax.bar(
            positions,
            values,
            width,
            yerr=errors,
            capsize=3,
            color=colors[descriptor],
            label=descriptor,
            error_kw={
                "elinewidth": 1.1,
                "capthick": 1.1,
            },
        )

        ax.bar_label(
            bars,
            labels=[f"{value:.1%}" for value in values],
            padding=3,
            fontsize=8,
            rotation=90,
        )

    ax.axhline(
        0.5,
        linestyle="--",
        color="black",
        linewidth=1,
        label="Random expectation",
    )

    ax.set_xticks(x, ["Top-5", "Top-10"])
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Mean same-topology neighbor fraction")
    ax.set_xlabel("Neighborhood size")
    ax.set_title("Topology enrichment among nearest neighbors")
    ax.legend(frameon=False, ncol=3)
    ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    return fig


fig = plot_neighbor_enrichment(results.neighbor_summary)

save_figure(
    fig,
    "same_topology_neighbor_enrichment.svg",
)

fig.savefig(
    "same_topology_neighbor_enrichment.png",
    dpi=300,
    bbox_inches="tight",
)

## 7. Expanded four-descriptor comparisons

These figures extend the matched-distance, within/across-library,
controlled-separation, and neighbor-enrichment analyses to ECFP, ECFP-Hg,
COAF, and POAF.


In [ ]:
from scipy.stats import gaussian_kde


DESCRIPTOR_COLORS = {
    "ECFP": "#f97316",
    "ECFP_Hg": "#059669",
    "COAF": "#3b82f6",
    "POAF": "#7c3aed",
}


def probability_contour_panel(ax, x, y, descriptor):
    """
    Plot probability-density contours for one descriptor against ECFP.
    """
    limit = max(float(np.max(x)), float(np.max(y))) * 1.03

    grid = np.linspace(0, limit, 150)
    grid_x, grid_y = np.meshgrid(grid, grid)

    density = gaussian_kde(
        np.vstack([x, y]),
        bw_method="scott",
    )(
        np.vstack([grid_x.ravel(), grid_y.ravel()])
    ).reshape(grid_x.shape)

    ordered = np.sort(density.ravel())[::-1]
    cumulative = np.cumsum(ordered) / ordered.sum()

    masses = (0.50, 0.75, 0.90, 0.95)

    thresholds = {
        mass: ordered[
            min(
                np.searchsorted(cumulative, mass),
                len(ordered) - 1,
            )
        ]
        for mass in masses
    }

    contour_colors = plt.cm.viridis(
        np.linspace(0.15, 0.90, len(masses))
    )

    for mass, color in zip(masses, contour_colors):
        threshold = thresholds[mass]

        contour = ax.contour(
            grid_x,
            grid_y,
            density,
            levels=[threshold],
            colors=[color],
            linewidths=1.7,
        )

        ax.clabel(
            contour,
            fmt={threshold: f"{int(100 * mass)}%"},
            inline=True,
            fontsize=8,
        )

    ax.plot(
        [0, limit],
        [0, limit],
        linestyle="--",
        color="black",
        linewidth=1,
    )

    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)
    ax.set_aspect("equal")

    ax.set_xlabel("Standard ECFP L1–L2 distance")
    ax.set_ylabel(f"{descriptor} L1–L2 distance")
    ax.set_title(f"{descriptor} versus ECFP")


def plot_added_matched_topology_contours(table):
    """
    Plot matched L1-L2 distance contours for ECFP_Hg, COAF, and POAF
    against standard ECFP.
    """
    descriptors = ("ECFP_Hg", "COAF", "POAF")

    required_columns = [
        "ECFP_distance_L1_L2",
        *[
            f"{descriptor}_distance_L1_L2"
            for descriptor in descriptors
        ],
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in table.columns
    ]

    if missing_columns:
        raise ValueError(
            "The matched-pair table is missing required columns: "
            f"{missing_columns}"
        )

    fig, axes = plt.subplots(
        1,
        len(descriptors),
        figsize=(16.5, 5),
    )

    x = table["ECFP_distance_L1_L2"].to_numpy()

    for ax, descriptor in zip(axes, descriptors):
        y = table[
            f"{descriptor}_distance_L1_L2"
        ].to_numpy()

        valid = np.isfinite(x) & np.isfinite(y)

        probability_contour_panel(
            ax,
            x[valid],
            y[valid],
            descriptor,
        )

    fig.tight_layout()
    return fig


fig = plot_added_matched_topology_contours(
    results.matched_pair_distances
)

save_figure(
    fig,
    "matched_topology_density_ECFP_Hg_COAF_and_POAF.svg",
)

fig.savefig(
    "matched_topology_density_ECFP_Hg_COAF_and_POAF.png",
    dpi=300,
    bbox_inches="tight",
)

## 8. Inspect the strongest descriptor differences

The first table lists matched IDs for which COAF increases the L1–L2 distance most strongly relative to ECFP. The second provides the counterexamples where ECFP gives the larger topology distance. Both are important for a balanced interpretation and for detecting systematic chemistry-specific behavior.

In [ ]:
def depict_ranked_topology_pairs(ranked_table, dataset, n=8, representation='COAF',
                                  mols_per_row=4, sub_image_size=(300, 230)):
    """Draw matched L1/L2 structures for the first n IDs in a ranked table.

    representation='COAF' retains the [Hg] root marker; representation='ECFP'
    shows the corresponding ordinary structures used for ECFP. Each ID produces
    two adjacent panels, one for L1 and one for L2.
    """
    representation = representation.upper()
    if representation not in {'COAF', 'ECFP'}:
        raise ValueError("representation must be 'COAF' or 'ECFP'")

    selected = ranked_table.head(n).copy()
    lookup = dataset.set_index('ID')
    molecules, legends = [], []
    for row in selected.itertuples(index=False):
        if row.ID not in lookup.index:
            continue
        source = lookup.loc[row.ID]
        delta = row.COAF_minus_ECFP_distance
        for topology in ('L1', 'L2'):
            smiles = source[f'{representation}_{topology}']
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                raise ValueError(f'Could not depict ID {row.ID}, {representation}_{topology}')
            molecules.append(mol)
            legends.append(f'{row.ID} | {topology} | Δ={delta:+.3f}')

    if not molecules:
        print('No ranked structures were available to depict.')
        return None
    return Draw.MolsToGridImage(
        molecules, molsPerRow=mols_per_row, subImgSize=sub_image_size,
        legends=legends, useSVG=True,
    )

print('Largest COAF advantage')
display(results.ranked_coaf_gain.round(4))
display(depict_ranked_topology_pairs(results.ranked_coaf_gain, dataset, n=8))

print('Largest ECFP advantage')
display(results.ranked_ecfp_favored.round(4))
display(depict_ranked_topology_pairs(results.ranked_ecfp_favored, dataset, n=8))

In [ ]:
from IPython.display import SVG, Markdown, display
from rdkit import Chem
from rdkit.Chem import Draw


def summarize_poaf_matched_distances(matched_table):
    """
    Summarize matched L1/L2 topology distances for POAF versus ECFP.
    """
    required_columns = [
        'ID',
        'ECFP_distance_L1_L2',
        'POAF_distance_L1_L2',
        'POAF_minus_ECFP_distance',
    ]

    missing = [
        column for column in required_columns
        if column not in matched_table.columns
    ]

    if missing:
        raise ValueError(
            f'Missing required columns: {missing}. '
            'Rerun the updated four-descriptor benchmark first.'
        )

    difference = matched_table[
        'POAF_minus_ECFP_distance'
    ].dropna()

    summary = pd.DataFrame({
        'metric': [
            'Number of matched compounds',
            'Mean ECFP L1–L2 distance',
            'Median ECFP L1–L2 distance',
            'Mean POAF L1–L2 distance',
            'Median POAF L1–L2 distance',
            'Mean POAF minus ECFP distance',
            'Median POAF minus ECFP distance',
            'Standard deviation of POAF minus ECFP',
            'Fraction with POAF distance > ECFP',
            'Fraction with ECFP distance > POAF',
            'Fraction with equal distances',
        ],
        'value': [
            len(matched_table),
            matched_table['ECFP_distance_L1_L2'].mean(),
            matched_table['ECFP_distance_L1_L2'].median(),
            matched_table['POAF_distance_L1_L2'].mean(),
            matched_table['POAF_distance_L1_L2'].median(),
            difference.mean(),
            difference.median(),
            difference.std(ddof=1),
            (difference > 1e-12).mean(),
            (difference < -1e-12).mean(),
            (difference.abs() <= 1e-12).mean(),
        ],
    })

    return summary


def select_illustrative_poaf_pairs(matched_table, n_pairs=6):
    """
    Select matched IDs with the largest positive and negative
    POAF-minus-ECFP topology-distance differences.
    """
    poaf_favored = (
        matched_table
        .nlargest(n_pairs, 'POAF_minus_ECFP_distance')
        .reset_index(drop=True)
    )

    ecfp_favored = (
        matched_table
        .nsmallest(n_pairs, 'POAF_minus_ECFP_distance')
        .reset_index(drop=True)
    )

    return poaf_favored, ecfp_favored


def display_poaf_matched_structure_pairs(
    ranked_table,
    dataset,
    title,
    n_pairs=6,
):
    """
    Display matched L1/L2 rooted structures side by side.

    The [Hg] marker is retained because it identifies the attachment
    point used by POAF.
    """
    selected = ranked_table.head(n_pairs).copy()

    structure_table = dataset.copy()
    structure_table['ID'] = structure_table['ID'].astype(str)
    lookup = structure_table.set_index('ID')

    molecules = []
    legends = []

    for rank, row in selected.reset_index(drop=True).iterrows():
        identifier = str(row['ID'])

        if identifier not in lookup.index:
            raise KeyError(
                f'ID {identifier} was not found in the input dataset.'
            )

        source = lookup.loc[identifier]

        metrics = (
            f"ECFP={row['ECFP_distance_L1_L2']:.3f}; "
            f"POAF={row['POAF_distance_L1_L2']:.3f}; "
            f"Δd={row['POAF_minus_ECFP_distance']:+.3f}"
        )

        for topology in ('L1', 'L2'):
            smiles = source[f'COAF_{topology}']
            molecule = Chem.MolFromSmiles(smiles)

            if molecule is None:
                raise ValueError(
                    f'Could not depict ID {identifier}, {topology}: '
                    f'{smiles}'
                )

            molecules.append(molecule)
            legends.append(
                f'Pair {rank + 1}: ID {identifier} | {topology}\n'
                f'{metrics}'
            )

    display(Markdown(f'### {title}'))

    drawing = Draw.MolsToGridImage(
        molecules,
        molsPerRow=2,
        subImgSize=(430, 280),
        legends=legends,
        useSVG=True,
    )

    # RDKit versions differ in whether they return an SVG string
    # or an IPython SVG display object.
    if isinstance(drawing, (str, bytes)):
        display(SVG(data=drawing))
    else:
        display(drawing)


# Summarize POAF behavior across the complete matched library.
print('POAF matched-topology summary')
display(
    summarize_poaf_matched_distances(
        results.matched_pair_distances
    ).round(4)
)


# Select examples at both extremes.
poaf_favored, ecfp_favored_relative_to_poaf = (
    select_illustrative_poaf_pairs(
        results.matched_pair_distances,
        n_pairs=6,
    )
)

table_columns = [
    'ID',
    'ECFP_distance_L1_L2',
    'POAF_distance_L1_L2',
    'POAF_minus_ECFP_distance',
]


print('Largest POAF-specific topology distances')
display(
    poaf_favored[table_columns].round(4)
)

display_poaf_matched_structure_pairs(
    poaf_favored,
    dataset,
    title='Largest POAF-specific topology-distance increases',
    n_pairs=6,
)


print('Largest ECFP-specific topology distances relative to POAF')
display(
    ecfp_favored_relative_to_poaf[table_columns].round(4)
)

display_poaf_matched_structure_pairs(
    ecfp_favored_relative_to_poaf,
    dataset,
    title='Counterexamples: ECFP distances larger than POAF',
    n_pairs=6,
)

## 9. Interpretation checklist

A compelling COAF result should be consistent across the three views rather than resting on one statistic:

- the matched `COAF_minus_ECFP` distance should be positive and practically meaningful;
- controlled cross-minus-within separation should be larger for COAF on the same sampled pairs;
- COAF should show higher same-topology neighbor enrichment, preferably across several *k* values;
- the advantage should not be explained solely by a large difference in fingerprint density; and
- the strongest examples and counterexamples should be chemically inspected.

Because this is a targeted benchmark, it should be included in the paper only if the finalized COAF shows a clear and reproducible advantage. The saved CSV files and `run_metadata.json` provide the numerical record needed for that decision.